# RAG Retrieval Accuracy — Profile Index

Measures how accurately the **profile-based FAISS index** retrieves training essays
whose label matches the query essay's true label, evaluated on the **val split**.

**Pipeline:**
1. Load `data/split/essays/val.csv` (247 rows).
2. Profile val essays label-blind → `data/profile_db/essays_val/` (skips already-done).
3. Embed each val profile with the text embedder (fallback to raw text if profiling failed).
4. For each trait and k ∈ {3, 5}: retrieve top-k from `data/vector_db/essays_profile/`.
5. Count how many retrieved samples share the query's true label (**label match rate**).
6. Save results to `result/retrieve_accuracy/profile/`.

**Metrics per (trait, k):**
- `mean_match_rate` — average fraction of retrieved samples whose label matches the query.
- `std_match_rate`  — standard deviation across queries.
- `hit_rate`        — fraction of queries where ≥ 1 retrieved sample matches.

In [1]:
import os, sys, time
from pathlib import Path
from typing import Dict

import numpy as np
import pandas as pd

project_root = Path.cwd().resolve()
if not (project_root / "ptd_model").exists():
    project_root = (project_root / ".." / "..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Project root:", project_root)

Project root: F:\std\GR\code\model_x_ocean


## Configuration

In [2]:
VAL_CSV        = str(project_root / "data/split/essays/val.csv")
VAL_PROFILE_DB = str(project_root / "data/profile_db/essays_val")
PROFILE_DB     = str(project_root / "data/vector_db/essays_profile")
RES_DIR        = str(project_root / "result/retrieve_accuracy/profile")

PROFILER_MODEL = "gpt-4o-mini"
K_VALUES       = [3, 5]

TRAITS = {
    "cOPN": "Openness to Experience",
    "cCON": "Conscientiousness",
    "cEXT": "Extraversion",
    "cAGR": "Agreeableness",
    "cNEU": "Neuroticism",
}

val_df = pd.read_csv(VAL_CSV)
print(f"Val split    : {len(val_df)} rows")
print(f"Profile DB   : {PROFILE_DB}")
print(f"Val profiles : {VAL_PROFILE_DB}")
print(f"Output       : {RES_DIR}")

for p in (PROFILE_DB,):
    if not Path(p).exists():
        raise RuntimeError(f"Missing: {p}. Run notebook/gpt/rag_profile_half1_embed.ipynb first.")

Val split    : 247 rows
Profile DB   : F:\std\GR\code\model_x_ocean\data\vector_db\essays_profile
Val profiles : F:\std\GR\code\model_x_ocean\data\profile_db\essays_val
Output       : F:\std\GR\code\model_x_ocean\result\retrieve_accuracy\profile


## Step 1 — Profile val essays (label-blind)

Profiles are generated **without exposing labels** (`use_labels=False`) to avoid leakage.
Already-profiled entries are skipped (append-safe).

In [3]:
from rag.profiler.store import ProfileStore
from rag.profiler.runner import build_profiles
from rag.profiler.prompts import FACETS

val_store_path = Path(VAL_PROFILE_DB) / "profile_store.jsonl"
val_store = ProfileStore(str(val_store_path))
val_store.load()

n_valid  = sum(1 for i in range(len(val_df)) if val_store.has(f"user_{i}") and val_store.get(f"user_{i}").get("valid"))
needed   = len(val_df) - n_valid
print(f"Val profiles in store : {len(val_store)}  (valid: {n_valid},  missing: {needed})")

if needed > 0:
    val_store = build_profiles(
        data       = val_df,
        output_dir = VAL_PROFILE_DB,
        model_name = PROFILER_MODEL,
        log_dir    = str(project_root / "log" / "profiler_val"),
        use_labels = False,
    )

val_profiles: Dict[int, dict] = {
    int(e["user_id"].split("_")[1]): e
    for e in val_store.get_all()
    if e.get("valid")
}
print(f"Val profiles ready    : {len(val_profiles)} / {len(val_df)}")

Val profiles in store : 0  (valid: 0,  missing: 247)
[20260512-001312] Profiling 247 essays -> F:\std\GR\code\model_x_ocean\data\profile_db\essays_val\profile_store.jsonl
[20260512-001312] Model: gpt-4o-mini  | use_labels=False
[20260512-001312] Logs : F:\std\GR\code\model_x_ocean\log\profiler_val\20260512-001312_log.txt
  [profiler] 10/247 done. errors=0 invalid=0 rate=0.06/s ETA=3695s
  [profiler] 20/247 done. errors=0 invalid=0 rate=0.06/s ETA=3665s
  [profiler] 30/247 done. errors=0 invalid=0 rate=0.05/s ETA=3967s
  [profiler] 40/247 done. errors=0 invalid=0 rate=0.05/s ETA=3825s
  [profiler] 50/247 done. errors=0 invalid=0 rate=0.05/s ETA=3896s
  [profiler] 60/247 done. errors=0 invalid=0 rate=0.05/s ETA=3562s
  [profiler] 70/247 done. errors=0 invalid=0 rate=0.06/s ETA=3214s
  [profiler] 80/247 done. errors=0 invalid=0 rate=0.06/s ETA=2913s
  [profiler] 90/247 done. errors=0 invalid=0 rate=0.06/s ETA=2726s
  [profiler] 100/247 done. errors=0 invalid=0 rate=0.06/s ETA=2513s
  [pro

## Step 2 — Load profile index & embed val queries

Each val query is embedded as its **full rendered profile text** (same representation
used to build the training index in `rag_profile_half1_embed.ipynb`).
Falls back to raw essay text for essays whose profiling failed.

In [4]:
from rag.embedder import get_embedding
from rag.faiss_index import FAISSIndex

profile_index = FAISSIndex(dimension=0)
profile_index.load(
    str(Path(PROFILE_DB) / "vectors.faiss"),
    str(Path(PROFILE_DB) / "vectors_meta.jsonl"),
)
print(f"Profile index loaded: {profile_index._index.ntotal} vectors, dim={profile_index.dimension}")

Profile index loaded: 1974 vectors, dim=768


In [5]:
def render_full_profile_text(entry: dict) -> str:
    """Render a profile entry to the same text format used by the training index."""
    raw = entry.get("raw") or ""
    if raw.strip():
        return raw
    facets = entry.get("facets", {})
    ling   = entry.get("linguistic", {})
    lines  = ["[FACETS]"]
    for code, name, *_ in FACETS:
        f = facets.get(code, {})
        lines.append(f"{code} {name:<18}| {f.get('signal','')} | {f.get('evidence','')}")
    lines.append("\n[LINGUISTIC]")
    for k, v in ling.items():
        lines.append(f"{k}: {v}")
    return "\n".join(lines)

In [6]:
query_texts    = []
used_profile   = []

for i, (_, row) in enumerate(val_df.iterrows()):
    pe = val_profiles.get(i)
    if pe:
        query_texts.append(render_full_profile_text(pe))
        used_profile.append(True)
    else:
        query_texts.append(str(row["text"]))
        used_profile.append(False)

n_profile_used = sum(used_profile)
print(f"Profile embedding : {n_profile_used} / {len(val_df)} essays")
print(f"Raw text fallback : {len(val_df) - n_profile_used} essays")

print("\nComputing batch embeddings ...")
t0 = time.time()
batch_vecs = get_embedding(query_texts)
query_embs = np.array(batch_vecs, dtype="float32")
print(f"Done in {time.time()-t0:.1f}s.  Shape: {query_embs.shape}")

Profile embedding : 247 / 247 essays
Raw text fallback : 0 essays

Computing batch embeddings ...
[embedder] Loading embedding model: nomic-ai/nomic-embed-text-v1.5


<All keys matched successfully>


Done in 315.0s.  Shape: (247, 768)


## Step 3 — Evaluate retrieval accuracy (one trait at a time, k = 3 and k = 5)

In [7]:
def normalize_label(val):
    s = str(val).strip().lower()
    if s in ("1", "1.0"): return "high"
    if s in ("0", "0.0"): return "low"
    return s


all_rows       = []
per_query_rows = []

for k in K_VALUES:
    print(f"\n{'='*55}")
    print(f"  k = {k}")
    print(f"{'='*55}")

    for trait_code, trait_full in TRAITS.items():
        match_rates, hits = [], []

        for i, (_, row) in enumerate(val_df.iterrows()):
            true_label = normalize_label(row[trait_code])
            if true_label not in ("high", "low"):
                continue

            # Retrieve k * 6 candidates, filter to those with a label for this trait
            _, results = profile_index.search(query_embs[i], k * 6)
            filtered   = [
                r for r in results
                if trait_full in r.get("trait_labels", {})
            ][:k]

            n       = len(filtered)
            matches = sum(
                1 for r in filtered
                if r["trait_labels"][trait_full] == true_label
            )
            rate = matches / n if n > 0 else 0.0
            hit  = int(matches > 0)

            match_rates.append(rate)
            hits.append(hit)
            per_query_rows.append({
                "query_idx":    i,
                "trait":        trait_full,
                "k":            k,
                "true_label":   true_label,
                "n_retrieved":  n,
                "match_count":  matches,
                "match_rate":   rate,
                "hit":          hit,
                "used_profile": used_profile[i],
            })

        mean_r = float(np.mean(match_rates))
        std_r  = float(np.std(match_rates))
        hit_r  = float(np.mean(hits))
        print(f"  {trait_full:<30s}  match_rate={mean_r:.4f} ± {std_r:.4f}  hit_rate={hit_r:.4f}")

        all_rows.append({
            "trait":           trait_full,
            "k":               k,
            "n_queries":       len(match_rates),
            "mean_match_rate": mean_r,
            "std_match_rate":  std_r,
            "hit_rate":        hit_r,
        })

summary_df   = pd.DataFrame(all_rows)
per_query_df = pd.DataFrame(per_query_rows)
print("\nEvaluation complete.")


  k = 3
  Openness to Experience          match_rate=0.5236 ± 0.3106  hit_rate=0.8583
  Conscientiousness               match_rate=0.5209 ± 0.3050  hit_rate=0.8704
  Extraversion                    match_rate=0.5209 ± 0.3137  hit_rate=0.8583
  Agreeableness                   match_rate=0.5466 ± 0.3413  hit_rate=0.8259
  Neuroticism                     match_rate=0.5574 ± 0.3462  hit_rate=0.8300

  k = 5
  Openness to Experience          match_rate=0.5368 ± 0.2692  hit_rate=0.9312
  Conscientiousness               match_rate=0.5012 ± 0.2656  hit_rate=0.9231
  Extraversion                    match_rate=0.5336 ± 0.2653  hit_rate=0.9352
  Agreeableness                   match_rate=0.5538 ± 0.2856  hit_rate=0.9433
  Neuroticism                     match_rate=0.5709 ± 0.2915  hit_rate=0.9312

Evaluation complete.


## Step 4 — Display & save results

In [8]:
print("=== Mean label match rate (fraction of retrieved samples matching query's true label) ===")
pivot = summary_df.pivot_table(
    index="trait", columns="k", values="mean_match_rate"
).rename_axis(columns="k")
display(pivot.round(4))

print("\n=== Hit rate (fraction of queries where ≥1 retrieved sample matches) ===")
pivot_hit = summary_df.pivot_table(
    index="trait", columns="k", values="hit_rate"
).rename_axis(columns="k")
display(pivot_hit.round(4))

=== Mean label match rate (fraction of retrieved samples matching query's true label) ===


k,3,5
trait,,
Agreeableness,0.5466,0.5538
Conscientiousness,0.5209,0.5012
Extraversion,0.5209,0.5336
Neuroticism,0.5574,0.5709
Openness to Experience,0.5236,0.5368



=== Hit rate (fraction of queries where ≥1 retrieved sample matches) ===


k,3,5
trait,,
Agreeableness,0.8259,0.9433
Conscientiousness,0.8704,0.9231
Extraversion,0.8583,0.9352
Neuroticism,0.8300,0.9312
Openness to Experience,0.8583,0.9312


In [ ]:
os.makedirs(RES_DIR, exist_ok=True)

summary_path   = os.path.join(RES_DIR, "accuracy_summary.csv")
per_query_path = os.path.join(RES_DIR, "per_query_details.csv")

summary_df.to_csv(summary_path, index=False)
per_query_df.to_csv(per_query_path, index=False)

print(f"Saved summary      -> {summary_path}")
print(f"Saved per-query    -> {per_query_path}")
print(f"\nRows in summary    : {len(summary_df)}")
print(f"Rows in per-query  : {len(per_query_df)}")

Saved summary      -> F:\std\GR\code\model_x_ocean\result\retrieve_accuracy\profile\accuracy_summary.csv
Saved per-query    -> F:\std\GR\code\model_x_ocean\result\retrieve_accuracy\profile\per_query_details.csv

Rows in summary    : 10
Rows in per-query  : 2470


: 